### Cosine Similarity with OpenAI Embeddings

Similarity Search

In [3]:
#Finding similar sentences
sentences=["The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"]


In [4]:
import numpy as np
def cosine_similarity(vec1,vec2):
    """
    Cosine similarity measures the angle between two vectors.
    -Result close to 1 : very similar
    -Result close to 0 : not related
    -Result close to -1 : Opposite meanings
    """

    dot_product=np.dot(vec1,vec2)
    mag_a=np.linalg.norm(vec1)
    mag_b=np.linalg.norm(vec2)
    return dot_product/(mag_a*mag_b)


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [7]:
sentence_embeddings=embeddings.embed_documents(sentences)
sentence_embeddings

[[0.1304018497467041,
  -0.011870170012116432,
  -0.028116997331380844,
  0.05123865231871605,
  -0.05597449094057083,
  0.03019159846007824,
  0.030161255970597267,
  0.02469838410615921,
  -0.018370557576417923,
  0.058766771107912064,
  -0.024953216314315796,
  0.06015418842434883,
  0.039831724017858505,
  0.03323051333427429,
  -0.06131140515208244,
  -0.04937310144305229,
  -0.05486353114247322,
  -0.04007603973150253,
  0.05642912536859512,
  0.03915659338235855,
  -0.034737102687358856,
  -0.013247678056359291,
  0.03196621686220169,
  -0.06349918246269226,
  -0.06017862260341644,
  0.07823450863361359,
  -0.028303824365139008,
  -0.047442857176065445,
  0.04035928100347519,
  -0.006630926392972469,
  -0.06674100458621979,
  -0.004191296175122261,
  -0.025311673060059547,
  0.05334164574742317,
  0.017428122460842133,
  -0.09792361408472061,
  0.006061288062483072,
  -0.06524167209863663,
  0.04557254910469055,
  0.023641789332032204,
  0.07658486068248749,
  -0.010264304466545

In [8]:
#Calculate the similarity between all pairs

for i in range(len(sentences)):
    for j in range(i+1,len(sentences)):
        similarity=cosine_similarity(sentence_embeddings[i],sentence_embeddings[j])
        print(f"{sentences[i]} vs {sentences[j]}")
        print(similarity)

The cat sat on the mat vs A feline rested on the rug
0.5643376629229653
The cat sat on the mat vs The dog played in the yard
0.1875860629607483
The cat sat on the mat vs I love programming in Python
0.02034990285746022
The cat sat on the mat vs Python is my favorite programming language
0.003496256281229553
A feline rested on the rug vs The dog played in the yard
0.27530906573754765
A feline rested on the rug vs I love programming in Python
0.06522404790597766
A feline rested on the rug vs Python is my favorite programming language
0.06269840481435698
The dog played in the yard vs I love programming in Python
0.12161608729268152
The dog played in the yard vs Python is my favorite programming language
0.09844987000828573
I love programming in Python vs Python is my favorite programming language
0.8773669004237147


Semantic Search

In [9]:
documents = [
    "LangChain is a framework for developing applications powered by language models",
    "Python is a high-level programming language",
    "Machine learning is a subset of artificial intelligence",
    "Embeddings convert text into numerical vectors",
    "The weather today is sunny and warm"
]
query="What is Langchain?"

In [10]:
def semantic_search(query,documents,embeddings_models,top_k=3):
    """Simple Sementic Search Implementation"""

    # top_k defines how many top sentences should be retrieved.

    #embed query and document
    query_embedding=embeddings_models.embed_query(query)
    doc_embeddings=embeddings_models.embed_documents(documents)

    #calculate the similarity score
    similarities=[]

    for i,doc_emb in enumerate(doc_embeddings):
        similarity=cosine_similarity(query_embedding,doc_emb)
        similarities.append((similarity,documents[i]))

    # sort by similarity
    similarities.sort(reverse=True) 
    #we get results in ascending order, but we need the result that matches the most first.
    return similarities[:top_k]

In [12]:
results=semantic_search(query,documents,embeddings)
results

[(np.float64(0.6586973106808739),
  'LangChain is a framework for developing applications powered by language models'),
 (np.float64(0.16037013724373644),
  'Python is a high-level programming language'),
 (np.float64(0.08682267409670372),
  'Machine learning is a subset of artificial intelligence')]

In [13]:
print(f"Semantic Search Results for : {query}")
for score,doc in results:
    print(f"{score:.3f}|{doc}")

Semantic Search Results for : What is Langchain?
0.659|LangChain is a framework for developing applications powered by language models
0.160|Python is a high-level programming language
0.087|Machine learning is a subset of artificial intelligence


In [14]:
query="What is embeddings"
results=semantic_search(query,documents,embeddings)
print(f"Semantic Search Results for : {query}")
for score,doc in results:
    print(f"{score:.3f}|{doc}")

Semantic Search Results for : What is embeddings
0.599|Embeddings convert text into numerical vectors
0.222|Machine learning is a subset of artificial intelligence
0.172|LangChain is a framework for developing applications powered by language models
